# 01 — Data exploration

This notebook introduces the BrainFlow streaming wrapper and basic **time-domain** visualization of EEG channels.

**Tip:** By default we use the **synthetic** BrainFlow board so you can run cells without hardware. For real SSVEP experiments, swap in your amplifier board id and a proper stimulus setup.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import numpy as np

from acquisition.brainflow_stream import BrainFlowStream
from utils.config import SSVEPConfig

In [ ]:
cfg = SSVEPConfig(project_root=ROOT)
cfg

## Stream a short segment (synthetic board)

We collect a few seconds of EEG after the stream starts, then plot the first channels.

In [ ]:
import time

stream = BrainFlowStream()
stream.prepare_session()
stream.start_stream()
time.sleep(2.0)
raw = stream.get_board_data()
stream.stop_stream()
stream.release_session()

meta = BrainFlowStream()
eeg_idx = meta.eeg_channel_indices()
fs = meta.sampling_rate()
eeg = raw[eeg_idx, :]

t = np.arange(eeg.shape[1]) / fs
plt.figure(figsize=(10, 4))
for ch in range(min(4, eeg.shape[0])):
    plt.plot(t, eeg[ch] + ch * 50e-6, label=f"EEG ch {ch}")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (+ offset)")
plt.legend()
plt.title("Synthetic board — short EEG snippet")
plt.tight_layout()
plt.show()

### Optional: save raw array

Use `numpy.save` or BrainFlow `DataFilter.write_file` for longer recordings once you have a labeling protocol.

In [ ]:
cfg.data_raw_dir.mkdir(parents=True, exist_ok=True)
np.save(cfg.data_raw_dir / "example_synthetic_window.npy", eeg)
print("Saved:", cfg.data_raw_dir / "example_synthetic_window.npy")